In [2]:
# ==========================================
# STEP 1: Dependencies & Environment Setup
# ==========================================
%pip install pypdf
from pathlib import Path
import nltk
from gensim.models import Word2Vec
from pypdf import PdfReader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 1.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Ensure necessary NLTK tokenizers are available locally
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
# Define and create output directory for the trained model weights
model_path = Path("../resources/trained_models/word2vec/pdf_word2vec.model")
model_path.parent.mkdir(parents = True, exist_ok = True)


In [5]:
# ==========================================
# STEP 2: Document Text Extraction
# ==========================================
# Initialize PDF reader for the target resume/document
reader = PdfReader("../resources/pdfs/data_science_01_david_kim.pdf")
raw_text = ""

# Iterate through all pages and extract text content sequentially
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        raw_text += page_text + " "  # Added trailing space to prevent word joining across pages





In [6]:
# ==========================================
# STEP 3: Text Preprocessing & Tokenization
# ==========================================
# Split the continuous text corpus into individual sentences
sentences = nltk.sent_tokenize(raw_text)

# Tokenize sentences into individual lowercase words to normalize the vocabulary
tokenized_sentences = [
    nltk.word_tokenize(sentence.lower()) for sentence in sentences
]

print("\nFirst 3 tokenized sentences for verification: \n", tokenized_sentences[:3], "\n\n")


First 3 tokenized sentences for verification: 
 [['resume', 'name', ':', 'david', 'kim', 'title', ':', 'data', 'scientist', 'category', ':', 'data', 'science', 'skills', 'python', ',', 'tensorflow', ',', 'pytorch', ',', 'scikit-learn', ',', 'sql', ',', 'pandas', ',', 'nlp', ',', 'statistics', 'experience', '4', 'years', 'building', 'predictive', 'models', 'and', 'recommendation', 'engines', '.'], ['published', 'research', 'on', 'deep', 'learning', '.'], ['education', 'bachelor', "'s", 'degree', 'in', 'a', 'field', 'related', 'to', 'data', 'science', '.']] 




In [7]:
# ==========================================
# STEP 4: Word2Vec Model Training & Saving
# ==========================================
# Train the Word2Vec model with the following architecture:
# - vector_size=50: Dimensionality of the word embeddings
# - window=5: Maximum distance between the current and predicted word within a sentence
# - min_count=1: Ignores all words with total frequency lower than this (set to 1 for small datasets)
model = Word2Vec(
    sentences = tokenized_sentences, 
    vector_size = 50, 
    window = 5, 
    min_count = 1
)

# Persist the trained model to disk for downstream tasks
model.save(str(model_path))

In [8]:
# ==========================================
# STEP 5: Model Evaluation & Vector Testing
# ==========================================
# Inspect the raw 50-dimensional vector representation of the keyword 'data'
print("Embedding vector for 'data':\n", model.wv["data"])

# Retrieve the top 10 most contextually similar words based on cosine similarity
print("\nWords most similar to 'data':")
for word, score in model.wv.most_similar("data", topn=10):
    print(f"{word}: {score:.4f}")

Embedding vector for 'data':
 [ 1.56395528e-02 -1.90216843e-02 -4.13717120e-04  6.93831779e-03
 -1.89057691e-03  1.67604890e-02  1.80238057e-02  1.30766081e-02
 -1.43240590e-03  1.54158035e-02 -1.70647092e-02  6.41756365e-03
 -9.27883759e-03 -1.01719191e-02  7.17803789e-03  1.07473731e-02
  1.55475708e-02 -1.15252770e-02  1.48606505e-02  1.32453591e-02
 -7.41395913e-03 -1.74859408e-02  1.08820312e-02  1.30184004e-02
 -1.57607731e-03 -1.34207904e-02 -1.41732972e-02 -4.98970551e-03
  1.02893617e-02 -7.33007956e-03 -1.87487155e-02  7.64758745e-03
  9.77329351e-03 -1.28613990e-02  2.41262442e-03 -4.15409263e-03
  5.23288654e-05 -1.97664965e-02  5.38040837e-03 -9.50295106e-03
  2.18199217e-03 -3.15658981e-03  4.38977126e-03 -1.57625172e-02
 -5.43253450e-03  5.32672461e-03  1.06880460e-02 -4.78123268e-03
 -1.90218519e-02  9.01373010e-03]

Words most similar to 'data':
engines: 0.2282
scientist: 0.2107
years: 0.2030
field: 0.1899
david: 0.1764
experience: 0.1670
resume: 0.1655
predictive: 0.1

In [ ]:
# 📘 Notes: Training Word2Vec from PDF

## Objective

Train a custom Word2Vec model using text extracted from a PDF and use it to find similar words.

---

## Workflow

```text
PDF
 ↓
Text Extraction
 ↓
Sentence Tokenization
 ↓
Word Tokenization
 ↓
Word2Vec Training
 ↓
Word Embeddings
 ↓
Similarity Search
```

---

## Libraries Used

### pypdf
Used to read PDF files and extract text.

### nltk
Used for:
- Sentence tokenization (`sent_tokenize`)
- Word tokenization (`word_tokenize`)

### gensim
Used to train and save the Word2Vec model.

---

## Text Preprocessing

### Sentence Tokenization

Converts text into sentences.

Example:

```text
Data science is fun. AI is powerful.
```

becomes

```python
[
    "Data science is fun.",
    "AI is powerful."
]
```

### Word Tokenization

Converts each sentence into individual words.

Example:

```text
Data Science is Fun
```

becomes

```python
["data", "science", "is", "fun"]
```

Lowercasing is applied to ensure consistent vocabulary.

---

## Word2Vec Training

```python
Word2Vec(
    sentences=tokenized_sentences,
    vector_size=50,
    window=5,
    min_count=1
)
```

### Parameters

| Parameter | Meaning |
|------------|----------|
| vector_size | Size of embedding vector |
| window | Number of neighboring words considered as context |
| min_count | Minimum occurrences required for a word to be included |

---

## What Word2Vec Learns

Word2Vec learns relationships between words based on surrounding context.

Example:

```text
King rules kingdom.
Queen rules kingdom.
```

The model learns that:

```text
King ≈ Queen
```

because both appear in similar contexts.

---

## Accessing Embeddings

Get embedding vector:

```python
model.wv["data"]
```

Find similar words:

```python
model.wv.most_similar("data")
```

---

## Important Observation

All words are converted to lowercase:

```python
sentence.lower()
```

Therefore:

```text
Deepankar → deepankar
```

Searching using uppercase may fail.

---

## Limitations

- Training data comes from a single PDF.
- Small datasets produce weaker embeddings.
- Vocabulary is limited to words present in the document.

---

## Word2Vec vs Sentence Transformers

| Feature | Word2Vec | Sentence Transformers |
|----------|----------|----------|
| Embedding Level | Word | Sentence |
| Training Required | Yes | No |
| Context Aware | No | Yes |
| Used in Modern RAG | Rarely | Very Common |

---

## Key Learnings

- Extract text from PDF
- Perform sentence tokenization
- Perform word tokenization
- Train a Word2Vec model
- Generate word embeddings
- Find similar words using vector similarity

---

## Interview Questions

**1. What is Word2Vec?**

An algorithm that learns vector representations of words from context.

**2. What does `window` do?**

Defines how many neighboring words are considered during training.

**3. What does `vector_size` represent?**

The number of dimensions in each word embedding.

**4. Why use `min_count`?**

To ignore infrequent words and reduce noise.

**5. Why are transformer embeddings preferred today?**

They capture context and semantic meaning much better than Word2Vec.